In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 1.9: Variational Autoencoder (Probabilistic Latent)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build a VAE with reparameterisation trick (z = mu + sigma * epsilon)
#   - Understand the ELBO loss: reconstruction + KL divergence
#   - Generate BRAND NEW images by sampling z ~ N(0, I)
#   - Visualise latent traversal to see what each dimension controls
#   - Apply to privacy-preserving synthetic patient data at NUH
#   - Verify synthetic data quality with statistical tests + privacy checks
#
# PREREQUISITES: 08_recurrent_ae.py
# ESTIMATED TIME: ~25 min
#
# TASKS:
#   1. Build VAE with mu/logvar heads and reparameterisation
#   2. Train with ELBO loss (reconstruction + KL divergence)
#   3. Visualise: reconstruction, generation, latent traversal
#   4. Apply: synthetic patient data for NUH PDPA compliance
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset



## THEORY — Probabilistic Latent Space


In [ ]:
# The VAE replaces the deterministic latent code with a PROBABILITY
# DISTRIBUTION. The encoder outputs mean (mu) and log-variance (log_var).
# We sample z ~ N(mu, sigma^2) using the reparameterisation trick:
#   z = mu + sigma * epsilon,  where epsilon ~ N(0, I)
# This keeps gradients flowing through mu and sigma.
#
# Loss = reconstruction + KL(q(z|x) || N(0,I))
# The KL term pushes the latent distribution toward a standard Gaussian,
# which is what allows GENERATION: sample z ~ N(0,I), decode to images.
#
# Analogy: A recipe book vs a cooking style guide. A regular AE stores
# exact recipes (deterministic codes). A VAE stores a STYLE GUIDE
# (probability distribution) — "use about this much salt, roughly this
# temperature." From the style guide, you can generate infinite new
# recipes that taste like the original chef's cooking.



## TASK 1 — Load data and engines


In [ ]:
X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader = load_fashion_mnist()
conn, tracker, exp_name, registry, has_registry = setup_engines()



## TASK 2 — Build and Train VAE


z = mu + sigma * epsilon. Gradients flow through mu and sigma.


Sample from the prior N(0, I) and decode to images.


VAE loss: reconstruction (MSE) + KL divergence.


In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int):
        super().__init__()
        # TODO: Build shared encoder body — nn.Sequential:
        #       Linear(input_dim, 256), ReLU, Linear(256, 128), ReLU
        self.encoder = ____

        # TODO: Two separate heads from the shared encoder output:
        #       fc_mu: Linear(128, latent_dim) — mean of q(z|x)
        #       fc_logvar: Linear(128, latent_dim) — log-variance of q(z|x)
        self.fc_mu = ____
        self.fc_logvar = ____

        # TODO: Build decoder — nn.Sequential:
        #       Linear(latent_dim, 128), ReLU, Linear(128, 256), ReLU,
        #       Linear(256, input_dim), Sigmoid
        self.decoder = ____

    def encode(self, x):
        # TODO: Pass x through encoder, return (mu, logvar)
        ____

    def reparameterise(self, mu, logvar):
        # TODO: std = exp(0.5 * logvar)
        #       eps = torch.randn_like(std)
        #       return mu + eps * std
        ____

    def forward(self, x):
        # TODO: encode -> reparameterise -> decode
        # Return (reconstruction, mu, logvar)
        ____

    def sample(self, n):
        # TODO: z = torch.randn(n, latent_dim, device=...)
        #       return self.decoder(z)
        ____


KL_WEIGHT = 0.1


def vae_loss_fn(model, xb):
    # TODO: Forward pass to get x_hat, mu, logvar
    # recon = F.mse_loss(x_hat, xb, reduction="sum") / xb.size(0)
    # kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / xb.size(0)
    # Return (recon + KL_WEIGHT * kl, {"recon": recon.item(), "kl": kl.item()})
    ____


print("\n" + "=" * 70)
print("  Variational AE — Probabilistic Latent Space")
print("=" * 70)
print("  Reparameterisation trick: z = mu + sigma * epsilon")
print(f"  KL weight: {KL_WEIGHT} (balance reconstruction vs regularity)")

# TODO: Create VAE(INPUT_DIM, LATENT_DIM) and train
vae_model = ____
vae_losses = ____



## TASK 3 — Visualise: Reconstruction, Generation, Latent Traversal


In [ ]:
# TODO: Three visualisations:
# show_reconstruction(vae_model, X_test_flat, "VAE Reconstruction")
# show_generated_samples(vae_model, "VAE — Generated Samples from N(0,I)", grid_size=8)
# show_latent_traversal(vae_model, X_test_flat, "VAE — Latent Traversal", n_dims=5)
____
____
____



### Checkpoint


In [ ]:
assert len(vae_losses) == EPOCHS
assert vae_losses[-1] < vae_losses[0]
vae_model.eval()
with torch.no_grad():
    samples = vae_model.sample(n=16).cpu()
assert samples.shape == (16, INPUT_DIM), f"Expected (16, 784), got {samples.shape}"
assert samples.min() >= 0.0 and samples.max() <= 1.0
print("\n--- Checkpoint passed --- VAE trained + generation verified\n")

if has_registry:
    register_model(registry, "vae", vae_model, vae_losses[-1])



## APPLY — Privacy-Preserving Synthetic Patient Data (NUH)


In [ ]:
# BUSINESS SCENARIO: You are a data scientist at National University
# Hospital (NUH). Researchers need patient data to study diabetes risk
# factors, but Singapore's PDPA prohibits sharing identifiable records.
# Your director asks: "Can we give researchers statistically useful
# data without exposing any real patient?"

print("\n" + "=" * 70)
print("  APPLICATION: PDPA-Compliant Synthetic Patient Data (NUH)")
print("=" * 70)

# --- Generate realistic patient data ---
N_PATIENTS = 10_000
patient_rng = np.random.default_rng(42)

# TODO: Generate correlated patient features:
# age ~ Normal(52, 15), clipped [21, 90]
# gender ~ Binomial(1, 0.48)
# bmi depends on age + gender + noise
# systolic_bp depends on age + bmi + noise
# diastolic_bp ~ 0.6 * systolic + noise
# cholesterol depends on age + bmi + noise
# hba1c depends on age + bmi + exponential noise
# glucose ~ 18 * hba1c + noise
# diagnosis ~ logistic(age, bmi, cholesterol, hba1c)
age = ____
gender = ____
bmi = ____
systolic_bp = ____
diastolic_bp = ____
cholesterol = ____
hba1c = ____
glucose = ____
diagnosis = ____

df = pl.DataFrame(
    {
        "age": age,
        "gender": gender,
        "bmi": bmi,
        "systolic_bp": systolic_bp,
        "diastolic_bp": diastolic_bp,
        "cholesterol": cholesterol,
        "hba1c": hba1c,
        "fasting_glucose": glucose,
        "diabetes_diagnosis": diagnosis,
    }
)
FEATURE_NAMES = df.columns
N_FEATURES = len(FEATURE_NAMES)
print(f"Patient dataset: {df.shape[0]:,} records, {N_FEATURES} features")
print(f"Diabetes prevalence: {diagnosis.mean()*100:.1f}%")

# --- Normalise and train VAE ---
data_np = df.to_numpy().astype(np.float32)
data_min = data_np.min(axis=0)
data_max = data_np.max(axis=0)
data_range = data_max - data_min
data_range[data_range == 0] = 1.0
data_norm = (data_np - data_min) / data_range

data_tensor = torch.tensor(data_norm, device=device)
patient_loader = DataLoader(TensorDataset(data_tensor), batch_size=256, shuffle=True)


class PatientVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=8):
        super().__init__()
        # TODO: Build encoder, mu/logvar heads, decoder
        # Encoder: Linear(input_dim, 64), ReLU, Linear(64, 32), ReLU
        # fc_mu: Linear(32, latent_dim)
        # fc_logvar: Linear(32, latent_dim)
        # Decoder: Linear(latent_dim, 32), ReLU, Linear(32, 64), ReLU,
        #          Linear(64, input_dim), Sigmoid
        self.encoder = ____
        self.fc_mu = ____
        self.fc_logvar = ____
        self.decoder = ____

    def encode(self, x):
        # TODO: Return (mu, logvar)
        ____

    def reparameterise(self, mu, logvar):
        # TODO: Reparameterisation trick
        ____

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        # TODO: encode -> reparameterise -> decode
        # Return (reconstruction, mu, logvar)
        ____


PATIENT_LATENT = 8
# TODO: Create PatientVAE, optimizer. Train 100 epochs with ELBO loss.
patient_model = ____
patient_opt = ____

print("\nTraining VAE on patient records...")
for epoch in range(100):
    patient_model.train()
    epoch_loss, n_samples = 0.0, 0
    for (batch,) in patient_loader:
        # TODO: Forward, compute recon_loss + kl_loss, backprop
        ____
    if (epoch + 1) % 25 == 0:
        print(f"  Epoch {epoch+1:3d}/100: loss = {epoch_loss/n_samples:.4f}")

# --- Generate synthetic patients ---
patient_model.eval()
with torch.no_grad():
    z = torch.randn(N_PATIENTS, PATIENT_LATENT, device=device)
    synthetic_norm = patient_model.decode(z).cpu().numpy()

synthetic_raw = synthetic_norm * data_range + data_min
synthetic_raw[:, 1] = np.round(synthetic_raw[:, 1]).clip(0, 1)
synthetic_raw[:, -1] = np.round(synthetic_raw[:, -1]).clip(0, 1)

print(f"\nGenerated {N_PATIENTS:,} synthetic patients")

# --- Visualisation 1: Distribution comparison ---
# TODO: Grid of histograms comparing real vs synthetic distributions
# Save to OUTPUT_DIR / "ex1_generation_distributions.png"
n_cols = 3
n_rows = (N_FEATURES + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3.5))
____
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_generation_distributions.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Visualisation 2: Correlation comparison ---
# TODO: 1x3 figure: real correlation, synthetic correlation, absolute difference
# Save to OUTPUT_DIR / "ex1_generation_correlations.png"
real_corr = np.corrcoef(data_np.T)
synth_corr = np.corrcoef(synthetic_raw.T)
corr_mae = np.abs(real_corr - synth_corr).mean()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
____
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_generation_correlations.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Privacy test ---
from scipy.spatial.distance import cdist

sample_synth = patient_rng.choice(N_PATIENTS, size=1000, replace=False)
sample_real = patient_rng.choice(N_PATIENTS, size=1000, replace=False)
dist_matrix = cdist(synthetic_norm[sample_synth], data_norm[sample_real])
nn_distances = dist_matrix.min(axis=1)
self_dist = cdist(data_norm[sample_real[:500]], data_norm[sample_real[500:]])
self_nn = self_dist.min(axis=1)
privacy_safe = nn_distances.mean() >= self_nn.mean() * 0.8

# --- Statistical utility ---
tests_passed = 0
for i in range(N_FEATURES):
    real_mean = data_np[:, i].mean()
    synth_mean = synthetic_raw[:, i].mean()
    real_std = data_np[:, i].std()
    if real_std > 0:
        passed = abs(synth_mean - real_mean) / real_std < 0.3
    else:
        passed = abs(synth_mean - real_mean) < 0.1
    if passed:
        tests_passed += 1

# --- Business Impact ---
print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — NUH PDPA-Compliant Synthetic Data")
print("=" * 64)
print(f"\nDataset: {N_PATIENTS:,} real -> {N_PATIENTS:,} synthetic records")
print(
    f"Statistical utility: {tests_passed}/{N_FEATURES} features pass ({tests_passed/N_FEATURES*100:.0f}%)"
)
print(f"Correlation MAE: {corr_mae:.4f}")
print(f"\nPrivacy assessment:")
print(f"  Mean synth-to-real NN distance: {nn_distances.mean():.4f}")
print(f"  Mean real-to-real NN distance:  {self_nn.mean():.4f}")
print(f"  Ratio (>1.0 = good):            {nn_distances.mean()/self_nn.mean():.3f}")
print(f"  Privacy safe: {'YES' if privacy_safe else 'NO'}")
print(f"\nResearch impact:")
print(f"  Before: 6-12 month ethics approval per data request")
print(f"  After: Instant access to synthetic data, ethics-exempt")
print(f"  Estimated: 3-5 research projects/year unblocked")
print(f"  Value: ~S$500K in grant revenue (S$100K avg per project)")
print("=" * 64)



## REFLECTION


[x] Built a VAE with reparameterisation trick (z = mu + sigma * eps)
  [x] Trained with ELBO loss (reconstruction + KL divergence)
  [x] Generated BRAND NEW images from the learned prior N(0,I)
  [x] Explored latent traversal — each dimension controls one aspect
  [x] Applied to synthetic patient data generation for NUH
  [x] Verified statistical utility AND privacy preservation

  KEY INSIGHT: The VAE trades reconstruction sharpness for a regular
  latent space. The KL term pushes q(z|x) toward N(0,I), which means
  you can sample z ~ N(0,I) and get plausible outputs. For synthetic
  data, this is the key: the generated records are statistically
  similar to real records but are NOT copies of any real patient.
  Singapore's PDPA is satisfied; researchers get useful data.

  Next: 10_contractive_vae.py combines smooth + probabilistic latent spaces...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments (VAE with KL term)


In [ ]:
# VAE-specific failure: posterior collapse (the encoder ignores x
# and outputs a fixed prior). It shows up in the Blood Test as
# gradients vanishing specifically on `fc_mu`/`fc_logvar` — the
# model stops routing information through the latent.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    xb = batch[0] if isinstance(batch, (tuple, list)) else batch
    loss, _ = vae_loss_fn(m, xb)
    return loss


print("\n── Diagnostic Report (VAE) ──")
diag, findings = run_diagnostic_checkpoint(
    vae_model,
    flat_loader,
    _diag_loss,
    title=f"VAE (KL={KL_WEIGHT})",
    n_batches=8,
    train_losses=vae_losses,
    show=False,
)

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
#   [!] Gradient flow (WARNING): Low gradients at
#       'fc_mu.weight' — RMS = 5.2e-05. KL penalty is
#       dampening mu-head updates (early sign of posterior
#       collapse risk). fc_logvar.weight RMS = 4.8e-05
#       (similar dampening).
#   [!] Dead neurons  (WARNING): 'decoder.2' (relu): 22%
#       dead during early epochs — sampled z lands far from
#       trained region.
#   [✓] Loss trend    (HEALTHY): total loss slope
#       -1.9e-03/epoch. Reconstruction term: -1.7e-03,
#       KL term: -2.1e-04. Both converging — no term
#       dominating.



## Final train loss: ~0.039 (recon 0.031 + KL 0.008), 10 epochs, beta=1.

STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:

 [BLOOD TEST — POSTERIOR-COLLAPSE DETECTOR] RMS 5.2e-05 on
    fc_mu.weight is the key VAE health metric. The KL term
    pulls q(z|x) toward N(0,I), which REDUCES gradient on
    the mean-encoder. If this drops below 1e-5, the encoder
    has given up and emits constant z regardless of input
    — POSTERIOR COLLAPSE. Slide 5M covers the Bowman 2016
    analysis: "the decoder ignores z and the encoder
    surrenders."
    >> Prescription: (a) KL annealing: start beta=0, linearly
       ramp to 1 over first 5 epochs. (b) Free-bits: cap KL
       loss from below at a small positive value (~0.5 per
       latent dim). (c) Reduce KL_WEIGHT below 1.0 if above.

 [X-RAY] 22% dead in decoder is an EARLY-EPOCH signature
    specific to VAEs: the sampled z ~ N(mu, sigma) lands in
    regions of latent space the decoder hasn't seen yet,
    triggering ReLU gates that never activated before.
    Usually recovers by epoch 5-6 as the decoder learns to
    cover the [-3, 3] sigma region of N(0,I).
    >> Prescription: If dead% STAYS above 15% past epoch 8,
       the reparameterisation sampling is too aggressive —
       clamp logvar to [-5, 5] to prevent sigma exploding.

 [STETHOSCOPE — TWO-TERM READ] VAE loss = reconstruction
    (pixelwise MSE / BCE) + KL divergence. Both SHOULD
    decrease. If total loss drops but KL rises: you have
    a regular AE pretending to be VAE (encoder ignoring KL).
    If KL drops to near zero and total stalls: posterior
    collapse (encoder ignoring input). The diag.epochs_df()
    exposes both terms — always plot them separately.
    >> Prescription: Watch both curves. Healthy VAE has
       reconstruction decreasing 5-10x faster than KL — the
       decoder learns first, then KL tightens the latent.

 FIVE-INSTRUMENT TAKEAWAY: VAE needs a two-part Stethoscope
 reading (recon + KL separately) because the sum can be
 healthy while each term is pathological. This is the first
 exercise where "total loss" lies — you will encounter the
 same pattern in ex_5 GANs (generator vs discriminator loss
 balance) and in ex_8 RL (policy vs value loss).


## TASK 3 — Visualise: Reconstruction, Generation, Latent Traversal


In [ ]:
show_reconstruction(vae_model, X_test_flat, "VAE Reconstruction")
show_generated_samples(vae_model, "VAE — Generated Samples from N(0,I)", grid_size=8)
show_latent_traversal(vae_model, X_test_flat, "VAE — Latent Traversal", n_dims=5)



### Checkpoint


In [ ]:
assert len(vae_losses) == EPOCHS
assert vae_losses[-1] < vae_losses[0]
vae_model.eval()
with torch.no_grad():
    samples = vae_model.sample(n=16).cpu()
assert samples.shape == (16, INPUT_DIM), f"Expected (16, 784), got {samples.shape}"
assert samples.min() >= 0.0 and samples.max() <= 1.0
print("\n--- Checkpoint passed --- VAE trained + generation verified\n")

if has_registry:
    register_model(registry, "vae", vae_model, vae_losses[-1])



## APPLY — Privacy-Preserving Synthetic Patient Data (NUH)


In [ ]:
# BUSINESS SCENARIO: You are a data scientist at National University
# Hospital (NUH). Researchers need patient data to study diabetes risk
# factors, but Singapore's PDPA prohibits sharing identifiable records.
# Your director asks: "Can we give researchers statistically useful
# data without exposing any real patient?"

print("\n" + "=" * 70)
print("  APPLICATION: PDPA-Compliant Synthetic Patient Data (NUH)")
print("=" * 70)

# --- Generate realistic patient data ---
N_PATIENTS = 10_000
patient_rng = np.random.default_rng(42)

age = (
    patient_rng.normal(loc=52, scale=15, size=N_PATIENTS)
    .clip(21, 90)
    .astype(np.float32)
)
gender = patient_rng.binomial(1, 0.48, size=N_PATIENTS).astype(np.float32)
bmi = (
    (22 + 0.05 * (age - 40) + 1.2 * gender + patient_rng.normal(0, 3.5, N_PATIENTS))
    .clip(16, 45)
    .astype(np.float32)
)
systolic_bp = (
    (100 + 0.4 * (age - 40) + 0.8 * (bmi - 24) + patient_rng.normal(0, 12, N_PATIENTS))
    .clip(85, 200)
    .astype(np.float32)
)
diastolic_bp = (
    (systolic_bp * 0.6 + patient_rng.normal(0, 8, N_PATIENTS))
    .clip(55, 120)
    .astype(np.float32)
)
cholesterol = (
    (150 + 0.5 * (age - 40) + 1.5 * (bmi - 24) + patient_rng.normal(0, 30, N_PATIENTS))
    .clip(100, 350)
    .astype(np.float32)
)
hba1c = (
    (
        5.0
        + 0.01 * (age - 40)
        + 0.05 * (bmi - 24)
        + patient_rng.exponential(0.4, N_PATIENTS)
    )
    .clip(4.0, 14.0)
    .astype(np.float32)
)
glucose = (
    (hba1c * 18 + patient_rng.normal(0, 15, N_PATIENTS))
    .clip(60, 300)
    .astype(np.float32)
)
diagnosis_logit = (
    -6.0
    + 0.03 * (age - 40)
    + 0.08 * (bmi - 24)
    + 0.005 * (cholesterol - 200)
    + 0.5 * (hba1c - 5.5)
)
diagnosis = patient_rng.binomial(1, 1 / (1 + np.exp(-diagnosis_logit))).astype(
    np.float32
)

df = pl.DataFrame(
    {
        "age": age,
        "gender": gender,
        "bmi": bmi,
        "systolic_bp": systolic_bp,
        "diastolic_bp": diastolic_bp,
        "cholesterol": cholesterol,
        "hba1c": hba1c,
        "fasting_glucose": glucose,
        "diabetes_diagnosis": diagnosis,
    }
)
FEATURE_NAMES = df.columns
N_FEATURES = len(FEATURE_NAMES)
print(f"Patient dataset: {df.shape[0]:,} records, {N_FEATURES} features")
print(f"Diabetes prevalence: {diagnosis.mean()*100:.1f}%")

# --- Normalise and train VAE ---
data_np = df.to_numpy().astype(np.float32)
data_min = data_np.min(axis=0)
data_max = data_np.max(axis=0)
data_range = data_max - data_min
data_range[data_range == 0] = 1.0
data_norm = (data_np - data_min) / data_range

data_tensor = torch.tensor(data_norm, device=device)
patient_loader = DataLoader(TensorDataset(data_tensor), batch_size=256, shuffle=True)


class PatientVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU()
        )
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterise(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        return self.decode(z), mu, logvar


PATIENT_LATENT = 8
patient_model = PatientVAE(N_FEATURES, PATIENT_LATENT).to(device)
patient_opt = torch.optim.Adam(patient_model.parameters(), lr=1e-3)

print("\nTraining VAE on patient records...")
for epoch in range(100):
    patient_model.train()
    epoch_loss, n_samples = 0.0, 0
    for (batch,) in patient_loader:
        recon, mu, logvar = patient_model(batch)
        recon_loss = F.mse_loss(recon, batch, reduction="sum")
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss = recon_loss + kl_loss
        patient_opt.zero_grad()
        loss.backward()
        patient_opt.step()
        epoch_loss += loss.item()
        n_samples += len(batch)
    if (epoch + 1) % 25 == 0:
        print(f"  Epoch {epoch+1:3d}/100: loss = {epoch_loss/n_samples:.4f}")

# --- Generate synthetic patients ---
patient_model.eval()
with torch.no_grad():
    z = torch.randn(N_PATIENTS, PATIENT_LATENT, device=device)
    synthetic_norm = patient_model.decode(z).cpu().numpy()

synthetic_raw = synthetic_norm * data_range + data_min
synthetic_raw[:, 1] = np.round(synthetic_raw[:, 1]).clip(0, 1)
synthetic_raw[:, -1] = np.round(synthetic_raw[:, -1]).clip(0, 1)

print(f"\nGenerated {N_PATIENTS:,} synthetic patients")

# --- Visualisation 1: Distribution comparison ---
n_cols = 3
n_rows = (N_FEATURES + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3.5))
axes_flat = axes.flatten()

for i, name in enumerate(FEATURE_NAMES):
    real_vals = data_np[:, i]
    synth_vals = synthetic_raw[:, i]
    if name in ("gender", "diabetes_diagnosis"):
        x = np.arange(2)
        real_counts = [np.mean(real_vals == c) for c in [0, 1]]
        synth_counts = [np.mean(synth_vals == c) for c in [0, 1]]
        axes_flat[i].bar(
            x - 0.15, real_counts, 0.3, label="Real", color="#2196F3", alpha=0.8
        )
        axes_flat[i].bar(
            x + 0.15, synth_counts, 0.3, label="Synthetic", color="#FF9800", alpha=0.8
        )
        axes_flat[i].set_xticks(x)
        axes_flat[i].set_xticklabels(
            ["Female", "Male"] if name == "gender" else ["No", "Yes"]
        )
    else:
        axes_flat[i].hist(
            real_vals, bins=40, alpha=0.6, density=True, label="Real", color="#2196F3"
        )
        axes_flat[i].hist(
            synth_vals,
            bins=40,
            alpha=0.6,
            density=True,
            label="Synthetic",
            color="#FF9800",
        )
    axes_flat[i].set_title(name.replace("_", " ").title(), fontsize=11)
    axes_flat[i].legend(fontsize=8)
for j in range(N_FEATURES, len(axes_flat)):
    axes_flat[j].set_visible(False)
fig.suptitle("Real vs VAE-Generated Patient Distributions", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_generation_distributions.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Visualisation 2: Correlation comparison ---
real_corr = np.corrcoef(data_np.T)
synth_corr = np.corrcoef(synthetic_raw.T)
corr_mae = np.abs(real_corr - synth_corr).mean()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, corr_mat, title in [
    (axes[0], real_corr, "Real Data"),
    (axes[1], synth_corr, "Synthetic Data"),
    (axes[2], np.abs(real_corr - synth_corr), "Absolute Difference"),
]:
    vmin = -1 if title != "Absolute Difference" else 0
    vmax = 1 if title != "Absolute Difference" else 0.3
    cmap = "RdBu_r" if title != "Absolute Difference" else "Reds"
    im = ax.imshow(corr_mat, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(N_FEATURES))
    ax.set_yticks(range(N_FEATURES))
    short_names = [n[:8] for n in FEATURE_NAMES]
    ax.set_xticklabels(short_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle(f"Correlation Preservation: MAE = {corr_mae:.4f}", fontsize=13)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_generation_correlations.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Privacy test ---
from scipy.spatial.distance import cdist

sample_synth = patient_rng.choice(N_PATIENTS, size=1000, replace=False)
sample_real = patient_rng.choice(N_PATIENTS, size=1000, replace=False)
dist_matrix = cdist(synthetic_norm[sample_synth], data_norm[sample_real])
nn_distances = dist_matrix.min(axis=1)
self_dist = cdist(data_norm[sample_real[:500]], data_norm[sample_real[500:]])
self_nn = self_dist.min(axis=1)
privacy_safe = nn_distances.mean() >= self_nn.mean() * 0.8

# --- Statistical utility ---
tests_passed = 0
for i in range(N_FEATURES):
    real_mean = data_np[:, i].mean()
    synth_mean = synthetic_raw[:, i].mean()
    real_std = data_np[:, i].std()
    if real_std > 0:
        passed = abs(synth_mean - real_mean) / real_std < 0.3
    else:
        passed = abs(synth_mean - real_mean) < 0.1
    if passed:
        tests_passed += 1

# --- Business Impact ---
print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — NUH PDPA-Compliant Synthetic Data")
print("=" * 64)
print(f"\nDataset: {N_PATIENTS:,} real -> {N_PATIENTS:,} synthetic records")
print(
    f"Statistical utility: {tests_passed}/{N_FEATURES} features pass ({tests_passed/N_FEATURES*100:.0f}%)"
)
print(f"Correlation MAE: {corr_mae:.4f}")
print(f"\nPrivacy assessment:")
print(f"  Mean synth-to-real NN distance: {nn_distances.mean():.4f}")
print(f"  Mean real-to-real NN distance:  {self_nn.mean():.4f}")
print(f"  Ratio (>1.0 = good):            {nn_distances.mean()/self_nn.mean():.3f}")
print(f"  Privacy safe: {'YES' if privacy_safe else 'NO'}")
print(f"\nResearch impact:")
print(f"  Before: 6-12 month ethics approval per data request")
print(f"  After: Instant access to synthetic data, ethics-exempt")
print(f"  Estimated: 3-5 research projects/year unblocked")
print(f"  Value: ~S$500K in grant revenue (S$100K avg per project)")
print("=" * 64)



## REFLECTION


[x] Built a VAE with reparameterisation trick (z = mu + sigma * eps)
  [x] Trained with ELBO loss (reconstruction + KL divergence)
  [x] Generated BRAND NEW images from the learned prior N(0,I)
  [x] Explored latent traversal — each dimension controls one aspect
  [x] Applied to synthetic patient data generation for NUH
  [x] Verified statistical utility AND privacy preservation

  KEY INSIGHT: The VAE trades reconstruction sharpness for a regular
  latent space. The KL term pushes q(z|x) toward N(0,I), which means
  you can sample z ~ N(0,I) and get plausible outputs. For synthetic
  data, this is the key: the generated records are statistically
  similar to real records but are NOT copies of any real patient.
  Singapore's PDPA is satisfied; researchers get useful data.

  Next: 10_contractive_vae.py combines smooth + probabilistic latent spaces...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

